In [ ]:
# Verificar o status da GPU
!nvidia-smi

In [1]:
import os

def listar_estrutura(pasta_raiz, nivel_max=3):
    print(f"📁 Estrutura de pastas em: {pasta_raiz}\n")
    for raiz, diretorios, arquivos in os.walk(pasta_raiz):
        nivel = raiz.replace(pasta_raiz, '').count(os.sep)
        if nivel >= nivel_max:
            continue
        indentacao = ' ' * 4 * (nivel)
        print(f"{indentacao}├── {os.path.basename(raiz)}/")
        sub_arquivos = [f for f in arquivos if not f.startswith('.')]
        if sub_arquivos:
            print(f"{indentacao}│   └── (Contém {len(sub_arquivos)} arquivos. Exemplo: {sub_arquivos[0]})")

# Executa para o FasterRCNN
if os.path.exists("/content/FasterRCNN_dataset"):
    listar_estrutura("/content/FasterRCNN_dataset")

if os.path.exists("/content/SSD_dataset"):
    listar_estrutura("/content/SSD_dataset")

📁 Estrutura de pastas em: /content/FasterRCNN_dataset

├── FasterRCNN_dataset/
    ├── annotations/
    │   └── (Contém 3 arquivos. Exemplo: train.json)
    ├── images/
        ├── test/
        │   └── (Contém 1034 arquivos. Exemplo: 000000501801.jpg)
        ├── val/
        │   └── (Contém 1033 arquivos. Exemplo: 000000325720.jpg)
        ├── train/
        │   └── (Contém 8265 arquivos. Exemplo: 000000094097.jpg)
📁 Estrutura de pastas em: /content/SSD_dataset

├── SSD_dataset/


In [1]:
import torch
import gc

# 1. Força a coleta de lixo do Python
gc.collect()

# 2. Esvazia o cache da GPU
torch.cuda.empty_cache()

# 3. Verifica se limpou de fato
print(f"Memória alocada na GPU: {torch.cuda.memory_allocated() / (1024**2):.2f} MB")

Memória alocada na GPU: 0.00 MB


In [2]:
import gc
import torch

# Deleta as variáveis pesadas se elas já existirem na memória global
if 'model' in globals():
    del model
if 'optimizer' in globals():
    del optimizer
if 'images' in globals():
    del images
if 'targets' in globals():
    del targets

# Força o coletor de lixo do Python e esvazia o cache retido pelo PyTorch na GPU
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# SSD dataset diagnostic
import torch

# Extrair apenas o primeiro lote (batch) do DataLoader de treino
images, targets = next(iter(train_loader))

print("🔍 === DIAGNÓSTICO DO DATASET PARA O SSD ===")

# Inspecionar apenas a primeira imagem deste lote
img = images[0]
target = targets[0]

print(f"Formato do Tensor da Imagem: {img.shape} (Deveria ser algo como [3, 300, 300])")
print(f"Valores dos Pixeis (Min - Max): {img.min().item():.4f} a {img.max().item():.4f}\n")

boxes = target['boxes']
labels = target['labels']

print(f"Total de objetos (bounding boxes) nesta imagem: {len(boxes)}")

if len(boxes) > 0:
    for i, (box, label) in enumerate(zip(boxes, labels)):
        xmin, ymin, xmax, ymax = box.tolist()
        
        print(f"\n📦 Objeto {i+1}:")
        print(f"   Classe (Label): {label.item()}")
        print(f"   Coordenadas: [xmin: {xmin:.2f}, ymin: {ymin:.2f}, xmax: {xmax:.2f}, ymax: {ymax:.2f}]")
        
        # --- VERIFICAÇÃO 1: Fundo (Background) ---
        if label.item() == 0:
            print("   ❌ ERRO CRÍTICO: Classe 0 detetada! No SSD do PyTorch, o 0 é reservado para o 'Background'. As classes dos seus objetos devem começar no 1.")
        else:
            print("   ✅ Classe parece correta (maior que 0).")
            
        # --- VERIFICAÇÃO 2: Formato YOLO vs Pascal VOC ---
        if xmin >= xmax or ymin >= ymax:
            print("   ❌ ERRO CRÍTICO: Coordenadas matematicamente impossíveis. O formato deve ser [x_min, y_min, x_max, y_max].")
            print("   Se estiver no formato YOLO [x_centro, y_centro, largura, altura], o SSD vai falhar.")
            
        # --- VERIFICAÇÃO 3: Escala (Normalizada vs Absoluta) ---
        elif xmax <= 1.01 and ymax <= 1.01:
            print("   ❌ ERRO CRÍTICO: As coordenadas estão normalizadas (entre 0 e 1). O SSD exige valores absolutos em píxeis (ex: 0 a 300).")
        else:
            print("   ✅ Coordenadas em escala absoluta e com limites lógicos (xmin < xmax).")
else:
    print("⚠️ Esta imagem de teste não tem anotações. Corra a célula novamente para testar outra imagem.")